# AICE Associate 방과후 · 4회차
## 데이터 전처리 ① — 이상치와 결측치 처리 (학생용)

| 항목 | 내용 |
|---|---|
| 일시 | 11월 11일(수) 16:00~18:00 |
| 시험 영역 | **데이터 전처리 (5~9번, 30점)** 중 오늘은 **5·6번 + 7번 앞부분** |
| 사용 데이터 | AICE 샘플 ① 회귀: `signal_data.csv` (내비게이션) / ② 분류: `wine_quality_data.csv` (와인) |

### 지난 시간에 찾은 '더러운 데이터' 🧹 → 오늘 청소합니다
| 데이터 | 발견한 것 | 오늘 쓰는 도구 |
|---|---|---|
| 내비게이션 | 평균시속 약 305km/h 행 | 조건 필터 `df[df.컬럼 < 300]` |
| 내비게이션 | Distance 결측치 | `isnull().sum()`, `dropna()` |
| 와인 | 와이너리 평점 이상치 (boxplot 수염 밖의 점) | IQR + `drop(...index)` |
| 와인 | E_FE_wine_kind, E_price 결측치 | `isnull()`, `dropna()` |
| 와인 | `red|rose|sparkling` 같은 복잡한 와인종류 값 | `isin()` + `loc`로 치환 |

### 오늘의 목표
1. `drop()`에서 **axis=0(행)과 axis=1(열)** 을 헷갈리지 않는다.
2. **조건 필터**와 **IQR**로 이상치를 지운다.
3. 결측치를 **세고(isnull) → 지우거나(dropna) 채운다(fillna)**.
4. 시험 5·6번을 빈칸형·직접작성형 모두 풀 수 있다.

> 🟢 = 모두 꼭 할 것, ★ = 빨리 끝난 사람 도전 과제

---
## 0. 준비

### 0-1. (Colab 전용) 한글 폰트 설치

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm
for path in fm.findSystemFonts():
    if 'Nanum' in path:
        fm.fontManager.addfont(path)
print('나눔 폰트 등록 완료')

### 0-2. 공통 실행 코드 (시험지 제공 코드)

In [ ]:
import platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

os_name = platform.system()
if os_name == 'Linux':
    plt.rc('font', family='NanumGothicCoding')
elif os_name == 'Windows':
    font = fm.FontProperties(fname='C:\\Windows\\Fonts\\gulim.ttc').get_name()
    mpl.rc('font', family=font)
elif os_name == 'Darwin':
    plt.rc('font', family='AppleGothic')
mpl.rcParams['axes.unicode_minus'] = False
print('준비 완료')

### 0-3. 🔁 복습: 3회차 1~4번 보지 않고 쓰기 (10분) 🟢
아래 4가지를 **보지 않고** 써 보세요. 막히면 3회차 치트시트를 봐도 돼요.
1. 내비게이션 데이터를 `df`에 읽고 첫 4행 출력
2. 주소1(Address1)의 countplot
3. 실주행시간(x)–평균시속(y) jointplot
4. 와인 데이터를 `data`에 읽고, 등급(x)별 와이너리 평점(y) boxplot

In [ ]:
# 복습 1~4

---
## 📋 시험의 '데이터 전처리' 영역 (5~9번, 30점)

| 번호 | 회귀 샘플 (내비게이션) | 분류 샘플 (와인) | 오늘? |
|---|---|---|---|
| 5 | 평균시속 ≥ 300 행 삭제 + RID 열 삭제 → `df_temp` | IQR로 와이너리 평점 이상치 삭제 + E_title 열 삭제 + 인덱스 초기화 → `data_temp` (빈칸) | ✅ 오늘 |
| 6 | 결측치 개수 → `답안6_1`, 결측 행 삭제 → `df_na` | 결측치 확인·삭제 → `data_na` (빈칸) | ✅ 오늘 |
| 7 | 시간 컬럼 삭제 + 원-핫 인코딩 (빈칸) | **와인종류 값 치환** + 컬럼 5개 삭제 + 원-핫 인코딩 (빈칸 + 오류정정) | ◐ 치환·삭제만 오늘 |
| 8 | train_test_split (80:20) | train_test_split (70:30, stratify) | 5회차 |
| 9 | RobustScaler (오류정정) | StandardScaler (오류정정) | 5회차 |

> 📌 **전처리 문항은 이어달리기예요.** `df → df_temp → df_na → df_preset`처럼 앞 문항의 결과 변수를 다음 문항이 그대로 씁니다. 5번에서 변수명을 틀리면 6번부터 줄줄이 무너져요. **변수명은 문제에 적힌 그대로!**

---
# 🧪 개념 교실 — 작은 표로 먼저 익히기 (20분)

진짜 데이터는 1만 행이라 결과가 눈에 안 들어와요. **5행짜리 표**로 먼저 연습하고 시험 문제로 가요.
각 셀을 실행하기 **전에** 결과를 먼저 예상해 보세요!

In [ ]:
mini = pd.DataFrame({
    'id':    ['a', 'b', 'c', 'd', 'e'],
    'speed': [30, 45, 305, 25, 50],
    'dist':  [3000, np.nan, 5000, 4000, np.nan],
    'city':  ['서울', '경기', '경기', '-', '서울']
})
mini

### ① drop — 행을 지울까, 열을 지울까?
| 쓰는 법 | 의미 |
|---|---|
| `mini.drop('id', axis=1)` | **열** 'id' 삭제 (axis=1 = 열 방향) |
| `mini.drop(columns=['id'])` | 위와 같음 (axis 안 써도 됨) |
| `mini.drop(2, axis=0)` | **행** 인덱스 2 삭제 (axis=0 = 행, 생략하면 기본값 0) |
| `mini.drop([0, 2])` | 행 인덱스 0, 2 삭제 |

> 🧠 외우는 법: **axis=0은 세로로 쭉 (행들을 따라), axis=1은 가로로 쭉 (열들을 따라)**. 헷갈리면 "**1**열"로 기억!

In [ ]:
# 예상해 보기: 결과 표에 몇 행, 몇 열이 남을까?
mini.drop('id', axis=1)

In [ ]:
mini.drop(2, axis=0)

### ② 원본이 바뀔까? — 재할당 vs inplace
`drop()`은 **원본을 바꾸지 않고 새 표를 돌려줘요.** 결과를 남기려면 둘 중 하나:
```python
mini2 = mini.drop('id', axis=1)          # 방법 1: 새 변수에 받기 (재할당)
mini.drop('id', axis=1, inplace=True)    # 방법 2: 원본을 직접 수정
```
⚠️ **절대 금지**: `mini = mini.drop('id', axis=1, inplace=True)` → inplace=True는 아무것도 돌려주지 않아서(**None**) mini가 사라져요!

In [ ]:
# 실험: 함정에 빠지면 어떻게 될까?
test = mini.copy()
test = test.drop('id', axis=1, inplace=True)
print(test)    # 예상해 보기!

### ③ 조건 필터 — 원하는 행만 남기기
`mini[조건]` → 조건이 True인 행만 남아요. **지우고 싶은 것의 반대**를 조건으로 쓰는 게 핵심!
- 시속 300 **이상을 지우고 싶다** → 300 **미만을 남긴다**: `mini[mini['speed'] < 300]`
- 조건 2개: `&`(그리고), `|`(또는), **각 조건을 괄호로** 감싸야 해요.

In [ ]:
print(mini['speed'] < 300)        # True/False 목록
mini[mini['speed'] < 300]

In [ ]:
# 조건 2개: 시속 300 미만 '그리고' 도시가 '-'가 아닌 행
mini[(mini['speed'] < 300) & (mini['city'] != '-')]

### ④ 결측치 — 세고, 지우고, 채우기
| 코드 | 의미 |
|---|---|
| `mini.isnull()` | 칸마다 결측이면 True |
| `mini.isnull().sum()` | **열별** 결측치 개수 |
| `mini.isnull().sum().sum()` | 전체 결측치 개수 (숫자 하나) |
| `mini.dropna()` | 결측치가 **하나라도 있는 행** 삭제 |
| `mini['dist'].fillna(값)` | 결측치를 값으로 채우기 |

> `isnull()`과 `isna()`는 완전히 같은 함수예요. 시험 빈칸에는 보통 `isnull`이 나와요.

In [ ]:
mini.isnull().sum()

In [ ]:
print('전체 결측치:', mini.isnull().sum().sum())
mini.dropna()

In [ ]:
# 채우기: 평균으로? 중앙값으로?
mini_fill = mini.copy()
mini_fill['dist'] = mini_fill['dist'].fillna(mini_fill['dist'].median())
mini_fill

#### ✍️ 개념 체크 (말로 답해 보기)
1. `mini.drop('speed')`를 실행하면 어떻게 될까요? 왜 그럴까요?
2. `mini['city']`의 `'-'`는 `isnull()`로 잡힐까요?
3. 최빈값으로 채우려고 `fillna(mini['city'].mode())`라고 썼는데 아무 일도 안 일어나요. 왜일까요?

---
# Part A. 회귀 샘플 — 내비게이션 데이터 청소

> 이 Part는 0-3 복습 셀에서 만든 `df`를 그대로 사용해요. 아직 안 했다면 아래 셀부터!

In [ ]:
df = pd.read_csv('./signal_data.csv')
print(df.shape)

### A-1. [시험 5번] 이상치 행 + 불필요 열 삭제 🟢
> jointplot으로 확인한 결과, 평균시속(Speed_Per_Hour) 컬럼의 이상치를 처리하려고 합니다.
> - 대상 데이터프레임 : df
> - 평균시속이 **300보다 크거나 같은** 이상치가 있는 행(row)을 삭제하세요.
> - AI모델링에 불필요한 경로ID(**RID**) 컬럼을 삭제하세요.
> - 전처리를 적용한 데이터프레임은 **df_temp**라는 변수명으로 저장하세요.

💡 "300 이상을 **지운다**" = "300 **미만을 남긴다**" 로 바꿔 생각하기

In [ ]:
# [5번] 여기에 답안코드를 작성하고 실행하세요.


print(df_temp.shape)   # 확인용

### A-2. [시험 6-1번] 결측치 개수 확인 🟢
> - 대상 데이터프레임 : df_temp
> - 결측치 개수를 확인하는 코드를 작성하세요.
> - 발견된 결측치의 개수를 **'답안6_1'** 변수에 저장해 주세요. (예시: `답안6_0 = 1234`)

In [ ]:
# [6-1번] 여기에 답안코드를 작성하고 실행하세요.


답안6_1 = 0

### A-3. [시험 6-2번] 결측치 행 삭제 🟢
> - 대상 데이터프레임 : df_temp
> - 결측치가 있는 행을 모두 삭제하세요.
> - 전처리가 반영된 데이터프레임은 변수명 **df_na**에 저장하세요.
> - 결측치가 사라졌는지 확인하세요.

In [ ]:
# [6-2번] 여기에 답안코드를 작성하고 실행하세요.

### A-4. 다른 방법: 지우지 않고 채우기 (fillna) 🟢
시험에서 "결측치를 **평균/중앙값/최빈값으로 대체**하세요"라고 나올 수도 있어요. 숫자 컬럼은 평균·중앙값, 글자 컬럼은 최빈값을 주로 써요.

In [ ]:
# df_temp를 복사한 df_fill에서 Distance 결측치를 '중앙값'으로 채우고, 결측치가 사라졌는지 확인하세요.
df_fill = df_temp.copy()

### A-5. (미리보기) 7번 앞부분 — 모델에 쓸 수 없는 열 지우기 🟢
> 원-핫 인코딩 전, **'Time_Departure', 'Time_Arrival'** 컬럼을 삭제해 주세요. (대상: df_na)

출발·도착 '시각' 문자열은 그대로는 모델이 못 써요(게다가 도착시각은 정답을 알려주는 반칙 정보!). **여러 열을 한 번에** 지울 땐 리스트로.

In [ ]:
# df_na에서 Time_Departure, Time_Arrival 두 컬럼을 삭제하세요. (inplace=True 사용)


df_na.info()

---
# Part B. 분류 샘플 — 와인 데이터 청소

In [ ]:
data = pd.read_csv('./wine_quality_data.csv')
print(data.shape)

### B-1. IQR로 이상치 기준선(fence) 구하기 — 개념 🟢
boxplot의 수염 끝이 바로 이 기준선이에요.

```
lower_fence = Q1 − 1.5 × IQR          upper_fence = Q3 + 1.5 × IQR
         ●  │──────┌────┬────┐──────│  ●
     이상치  lower   Q1  중앙값  Q3   upper  이상치
                    └── IQR ──┘
```
- `quantile(0.25)` = Q1 (아래에서 25% 지점), `quantile(0.75)` = Q3

In [ ]:
q1 = data['FE_points_winery'].quantile(0.25)
q3 = data['FE_points_winery'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print('Q1:', q1, ' Q3:', q3, ' IQR:', iqr)
print('lower_fence:', lower_fence, ' upper_fence:', upper_fence)

📌 3회차 boxplot(등급별)과 숫자가 조금 다를 수 있어요. 시험 5번은 **등급을 나누지 않고 전체** 와이너리 평점으로 기준선을 구하거든요.

### B-2. [시험 5번] 이상치 행 삭제 + 열 삭제 + 인덱스 초기화 (빈칸형) 🟢
> - 대상 데이터프레임 : data
> - 와이너리 평점이 upper_fence보다 **큰** / lower_fence보다 **작은** 이상치 행을 삭제하세요.
> - AI모델링에 불필요한 와인이름(**E_title**) 컬럼을 삭제하세요.
> - 전처리를 적용한 데이터프레임은 **data_temp**로 저장하고, 완료한 뒤 **index를 초기화**해 주세요.
> - `< blank5-1 >`, `< blank5-2 >`에 들어갈 내용을 '답안5_1', '답안5_2'에 저장하세요.

In [ ]:
# [5번] 아래 코드의 빈칸을 채우고 실행하세요.

q1 = data['FE_points_winery'].quantile(0.25)
q3 = data['FE_points_winery'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

data_temp = data.< blank5-1 >( data[(data['FE_points_winery'] > upper_fence) | (data['FE_points_winery'] < lower_fence)].< blank5-2 >, axis=0)
data_temp = data_temp.drop( columns=['E_title'])
data_temp.reset_index(drop=True, inplace=True)


답안5_1 = ''
답안5_2 = ''

#### 🔍 코드 한 줄 해부
```python
data.drop(  data[ (조건1) | (조건2) ].index  , axis=0)
           └──── 지울 행들만 골라낸 표 ────┘
                                     └ 그 행들의 번호 ┘
└ 그 번호의 행을 지워라 ┘
```
- `|` = 또는: 위로 벗어났거나 **또는** 아래로 벗어난 행
- `reset_index(drop=True)`: 행을 지우면 번호가 0,1,3,7...처럼 듬성듬성 → 0부터 다시 매김. `drop=True`가 없으면 옛 번호가 **새 열(index)로 추가**돼 버려요.

### B-3. [시험 6번] 결측치 확인 → 삭제 (빈칸형) 🟢
> - 대상 데이터프레임 : data_temp
> - 결측치 개수를 확인한 후에 결측치가 있는 행을 모두 삭제하세요.
> - 전처리가 반영된 데이터프레임은 **data_na**에 저장하고, 결측치가 사라졌는지 확인하세요.
> - `< blank6-1 >`, `< blank6-2 >`에 들어갈 내용을 '답안6_1', '답안6_2'에 저장하세요.

In [ ]:
# [6번] 아래 코드의 빈칸을 채우고 실행하세요.

print( '결측치 처리 전\n', data_temp.< blank6-1 >().sum() )
data_na = data_temp.< blank6-2 >()
print( '결측치 처리 후\n', data_na.< blank6-1 >().sum() )


답안6_1 = ''
답안6_2 = ''

### B-4. [시험 7번 앞부분] 와인종류 값 정리 — isin + loc 🟢
3회차 countplot에서 와인종류가 `white|sparkling`, `red|rose|sparkling` 등 12가지로 너무 지저분했죠? 4가지(red / white / rose / sparkling)로 정리해요.

> - 대상 데이터프레임 : data_na
> - `'white|rose|sparkling'`, `'red|white|sparkling'`, `'red|white|rose|sparkling'`, `'red|sparkling'` → **'sparkling'**
> - `'white|sparkling'` → **'white'**
> - `'red|rose'`, `'red|white'`, `'red|rose|sparkling'` → **'red'**
> - 치환 후 와인종류의 카테고리별 개수를 출력하세요.

📌 `isin([목록])` = 값이 목록 안에 **있으면 True**. `loc[행 조건, 열] = 새 값` = 조건에 맞는 행의 그 열만 바꿔요.

In [ ]:
# 먼저 작은 예로 확인
s = pd.Series(['red', 'red|rose', 'white', 'red|white'])
print(s.isin(['red|rose', 'red|white']))

In [ ]:
# [7번 앞부분] 빈칸 < blank7-1 >, < blank7-2 >를 채우고 실행하세요.

data_na.loc[data_na['E_FE_wine_kind'].< blank7-1 >(['white|rose|sparkling','red|white|sparkling', 'red|white|rose|sparkling', 'red|sparkling']), ['E_FE_wine_kind']] = 'sparkling'
data_na.loc[data_na['E_FE_wine_kind'].< blank7-1 >(['white|sparkling']), ['E_FE_wine_kind']] = 'white'
data_na.loc[data_na['E_FE_wine_kind'].< blank7-1 >(['red|rose', 'red|white', 'red|rose|sparkling']), ['E_FE_wine_kind']] = 'red'
print(data_na['E_FE_wine_kind'].< blank7-2 >())


# data_na의 와인종류 중 'red'의 개수를 '답안7'에 저장
답안7 = 0

### B-5. [시험 7번 앞부분] 불필요 컬럼 5개 삭제 (오류 정정형) 🟢
> 불필요한 컬럼 5개('E_province', 'E_region_1', 'E_region_2', 'E_winery', 'E_wine_variety')를 삭제하세요.
> 아래 코드에는 오류가 **1군데** 있어요.

In [ ]:
# 아래의 코드를 정정하고 실행하세요.

data_na = data_na.drop( ['E_province', 'E_region_1', 'E_region_2', 'E_winery', 'E_wine_variety'], axis=0)
data_na.info()

---
# Part C. 시험형 연습 문제 (자동 채점)

채점 셀을 먼저 실행하세요. `채점('C1', 답안C1)`처럼 쓰면 돼요.

In [ ]:
# 채점 셀 — 실행만 하세요
import hashlib
_정답 = {"C1": "aab3238922bcc25a6f606eb525ffdc56", "C2_1": "978d76676f5e7918f81d28e7d092ca0d", "C2_2": "a666587afda6e89aec274a3657558a27", "C3": "d3d9446802a44259755d38e6d163e820", "C4": "03fcd68e5673f08be96d2b6bb5be8261", "C5_1": "0d9959e8150fc305808d8838e888b622", "C5_2": "38ad6fe465c71d3a30b0d1a94766e8b0", "C6": "e2459ae2cf7921589685a6835ec08a7c"}
def 채점(번호, 답):
    h = hashlib.md5(str(답).strip().replace(' ', '').lower().encode()).hexdigest()
    print(f'[{번호}]', '⭕ 정답!' if h == _정답.get(번호) else '❌ 다시 생각해 보세요')
print('채점 준비 완료')

### C1. (직접 작성형) 🟢
> 대상 데이터프레임 : **df** (원본)
> 평균시속(Speed_Per_Hour)이 **100보다 크거나 같은** 행은 몇 개인지 구해 '답안C1'에 숫자로 저장하세요.
> 💡 조건 필터 결과의 `shape[0]` 또는 `len()`

In [ ]:
# [C1] 여기에 답안코드를 작성하고 실행하세요.


답안C1 = 0
채점('C1', 답안C1)

### C2. (직접 작성형 — IQR) ★
> 대상 데이터프레임 : **df** (원본)
> 실주행시간(Time_Driving)에 IQR 방법을 적용하려고 합니다.
> - upper_fence(Q3 + 1.5×IQR)를 구하고, **반올림한 정수**를 '답안C2_1'에 저장하세요. (`round()` 사용)
> - Time_Driving이 upper_fence보다 **큰** 행의 개수를 '답안C2_2'에 저장하세요.

In [ ]:
# [C2] 여기에 답안코드를 작성하고 실행하세요.


답안C2_1 = 0
답안C2_2 = 0
채점('C2_1', 답안C2_1)
채점('C2_2', 답안C2_2)

### C3. (오류 정정형) 🟢
> 원본 df에서 RID와 Time_Departure 컬럼을 삭제한 df_c를 만들고, 결측치 개수를 확인하려 합니다. 오류 **2군데**를 고친 뒤, df_c의 **컬럼 개수**를 '답안C3'에 저장하세요.

In [ ]:
# [C3] 아래의 코드를 정정하고 실행하세요.

df_c = df.drop(['RID', 'Time_Departure'], axis=0)
print(df_c.isnull.sum())


답안C3 = 0
채점('C3', 답안C3)

### C4. (직접 작성형 — fillna) 🟢
> 원본 df를 복사한 df_f에서 Distance 컬럼의 결측치를 **Distance의 중앙값**으로 채우세요.
> 채운 값(중앙값)을 정수로 '답안C4'에 저장하세요.

In [ ]:
# [C4] 여기에 답안코드를 작성하고 실행하세요.
df_f = df.copy()


답안C4 = 0
채점('C4', 답안C4)

### C5. (빈칸 — 와인) 🟢
> B-4에서 와인종류 값을 바꿀 때 쓴 빈칸 두 개, `< blank7-1 >`과 `< blank7-2 >`에 들어간 함수 이름을 각각 '답안C5_1', '답안C5_2'에 저장하세요. (괄호 없이 이름만)

In [ ]:
# [C5]
답안C5_1 = ''
답안C5_2 = ''
채점('C5_1', 답안C5_1)
채점('C5_2', 답안C5_2)

### C6. (결과 확인 — 와인) 🟢
> B-3의 **결측치 처리 전** 출력에서, 가격(**E_price**) 컬럼의 결측치 개수를 '답안C6'에 저장하세요.
> (B-2의 5번을 정확히 풀어야 맞출 수 있어요!)

In [ ]:
# [C6]
답안C6 = 0
채점('C6', 답안C6)

### ★ 도전 과제
1. 내비게이션 Address1의 **가짜 결측치 `'-'`** 를 `replace('-', np.nan)`으로 진짜 결측치로 바꾼 뒤, `isnull().sum()`으로 몇 개인지 확인하고 삭제해 보세요.
2. 원본 df에서 **평균시속 300 미만이면서 거리(Distance)가 결측이 아닌** 행만 한 줄로 남겨 보세요. (힌트: `&`, `notnull()`)
3. C5의 isin+loc 치환을 **`replace`** 로 다시 써 보세요.

In [ ]:
# ★ 도전 과제

---
# 📝 오늘의 정리

### 전처리 ① 치트시트
| 하고 싶은 것 | 코드 |
|---|---|
| 열 삭제 | `df.drop('열', axis=1)` · `df.drop(columns=['열1', '열2'])` |
| 행 삭제 (번호로) | `df.drop(행번호들, axis=0)` |
| 조건으로 이상치 행 삭제 | `df[df['열'] < 기준]` (남길 것을 조건으로) |
| 조건으로 이상치 행 삭제 (drop 방식) | `df.drop(df[지울조건].index, axis=0)` |
| IQR 기준선 | `q1 = quantile(0.25)`, `q3 = quantile(0.75)`, `iqr = q3-q1`, `q1-1.5*iqr`, `q3+1.5*iqr` |
| 인덱스 다시 매기기 | `df.reset_index(drop=True, inplace=True)` |
| 결측치 개수 | `df.isnull().sum()` (열별) · `.sum().sum()` (전체) |
| 결측 행 삭제 | `df.dropna()` |
| 결측치 채우기 | `df['열'] = df['열'].fillna(df['열'].median())` · 최빈값은 `mode()[0]` |
| 값 바꾸기 | `df.loc[df['열'].isin([...]), '열'] = '새값'` |
| 값별 개수 | `df['열'].value_counts()` |

### 시험 함정 체크리스트
- [ ] 열을 지울 땐 **axis=1** (또는 `columns=`)
- [ ] `inplace=True`를 쓰면 **앞에 `변수 =`를 붙이지 않는다**
- [ ] 조건 2개는 `(조건1) & (조건2)`, **괄호 필수**, `and`/`or` 말고 `&`/`|`
- [ ] "300 이상 삭제" = "300 **미만** 남기기"
- [ ] `reset_index(drop=True)` — drop=True를 빼먹지 않는다
- [ ] `isnull()`, `index` — 괄호가 필요한 것(함수)과 아닌 것(속성) 구분
- [ ] 문제가 말한 **변수명**(df_temp, df_na, data_temp, data_na)으로 저장
- [ ] "확인하세요"라고 하면 **확인 코드까지** 작성

### 다음 시간 (5회차, 11/16 월) 예고 — 전처리 ②
오늘 만든 `df_na`, `data_na`를 이어서 **원-핫 인코딩(get_dummies) → train/valid 분리(train_test_split) → 스케일링**까지 하면 모델에 넣을 준비 끝! 시험 7·8·9번이에요.